# Retail Intelligence: Sales & Profitability Exploratory Data Analysis

**Business Objective:** How can a retail business improve profitability without sacrificing sales growth?

**Analyst Background & Methodology:**
- **Dataset:** Tableau Sample - Superstore (Educational benchmark covering 9,994 transactional line items across US retail orders from 2014 to 2017).
- **Analytical Standards:** Distinct orders counted using unique order IDs; overall margins aggregated as $\frac{\sum \text{Profit}}{\sum \text{Sales}}$; sample sizes explicitly reported; correlation distinguished from causation.


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import FIGURES_DIR, PROCESSED_DATA_FILE
from src.metrics import executive_kpi_summary
from src.visualization import (
    plot_category_profitability,
    plot_customer_deciles,
    plot_discount_band_analysis,
    plot_monthly_trends,
    plot_regional_profitability,
)

print("Libraries and project modules loaded successfully.")

## 1. Load and Verify Cleaned Data
We load the verified cleaned dataset created by our pipeline in Stage 2.

In [ ]:
df = pd.read_csv(PROCESSED_DATA_FILE)
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date Range: {df['order_date'].min().date()} to {df['order_date'].max().date()} ({df['order_year'].nunique()} calendar years)")
df[["order_id", "order_date", "customer_name", "category", "sub_category", "sales", "discount", "profit"]].head()

## 2. Executive KPI Summary Scorecard
Here we compute the core baseline metrics across all 9,994 transactions.

In [ ]:
kpis = executive_kpi_summary(df)
kpi_df = pd.DataFrame(list(kpis.items()), columns=["Metric", "Value"])
display(kpi_df)

### Analytical Takeaway:
- The company generated **$2,297,200.86** in net revenue and **$286,397.02** in profit (12.47% margin).
- **The core vulnerability:** The business leaked **-$156,131.29** across 1,871 loss-making line items (18.72% of all items), affecting 1,318 distinct orders (26.31% of all orders).

## 3. Monthly Sales & Profit Trajectory (Seasonality Analysis)
We inspect monthly performance over the 48-month span to detect seasonality patterns and margin stability during volume spikes.

In [ ]:
fig_monthly = plot_monthly_trends(df)
plt.show()

### Observations:
1. **Strong Q4 Seasonality:** Significant revenue surges occur every November and December (holiday shopping peaks).
2. **Margin Compression during Peaks:** In November 2017, sales reached an all-time monthly high of $118.4K, but net profit margin dropped to **8.18%** (compared to the 12.47% baseline), showing that holiday volume was driven by aggressive margin concessions.

## 4. Category & Subcategory Profitability (Identifying Loss Leaders)
We examine all 17 subcategories to distinguish high-margin growth categories from persistent loss drains.

In [ ]:
fig_cat = plot_category_profitability(df)
plt.show()

### Observations:
- **Technology** is the primary profit powerhouse, delivering over 50% of company profit (Copiers: +$55.6K, 37.2% margin; Phones: +$44.5K, 13.5% margin).
- **Furniture** is severely margin-diluted (2.49% overall margin), dragged down by **Tables (-$17.7K net loss)** and **Bookcases (-$3.5K net loss)**.
- **Supplies** in Office Supplies is also net unprofitable (-$1.2K loss, -2.55% margin).

## 5. Discount Policy Analysis: The Observed 20% Breakeven Cliff
We evaluate overall margin across discrete discount bands and inspect sample sizes to ensure fair comparisons.

In [ ]:
fig_disc = plot_discount_band_analysis(df)
plt.show()

### Observations & Nuances:
- In this dataset, transactions with **0% to 20% discount** remain resiliently profitable (margins between +11.6% and +29.8%).
- Beyond 20% discount, profitability collapses:
  - **20.1% - 30%:** -10.05% margin (74% of items unprofitable)
  - **30.1% - 50%:** -24.80% margin (90% of items unprofitable)
  - **> 50%:** -119.17% margin (100% of items unprofitable, losing $1.19 per dollar of net sales)
- **Important Caveat:** This 20% threshold is an observed empirical pattern in this specific retailer's cost structure, not an immutable law of commerce.

## 6. Regional & Geographic Profit Drains
We examine regional margins and pinpoint the top states where heavy promotional discounting destroys bottom-line value.

In [ ]:
fig_reg = plot_regional_profitability(df)
plt.show()

### Observations:
- **West Region:** Delivers highest profit ($108.4K) and highest margin (14.94%).
- **Central Region:** Has the lowest margin (7.92%) and highest weighted discount rate (15.52%).
- **State Loss Leaders:** Texas (-$25.7K loss, 37.0% avg discount), Pennsylvania (-$15.6K loss, 32.9% avg discount), and Illinois (-$12.6K loss, 38.9% avg discount) are the primary geographic loss centers.

## 7. Customer Value Concentration (Pareto Decile Analysis)
We segment all 793 customers into 10 profit deciles to evaluate whether top buyers or bottom buyers drive company economics.

In [ ]:
fig_cust = plot_customer_deciles(df)
plt.show()

### Observations:
- **Pareto 80/20 Rule:** The top 20% of customers (Deciles 1 and 2, 160 customers) generate **$234.6K in profit**, representing **81.9%** of total company earnings.
- **Decile 10 Anomaly:** The bottom 10% of customers (79 customers) generate $278.3K in gross sales (a substantial volume of business), but incur **-$65,753.64 in cumulative net losses** due to an average discount rate exceeding 37%.
- **Strategic Takeaway:** The business does not need to abandon these customers; rather, it needs to re-align discount thresholds and enforce minimum order margins for high-volume, discount-exploiting accounts.

## 8. Evidence-Backed Business Recommendations

| Strategic Pillar | Observed Evidence | Recommended Action | Expected Business Impact |
| :--- | :--- | :--- | :--- |
| **1. Promotional Guardrails** | Transactions discounted >20% suffer aggregate negative margins (-10% to -119%), accounting for over $135K in cumulative losses. | Cap discretionary sales discounts at 20%; require senior VP margin approval for any discount >20%. | Protects margin without sacrificing volume for standard price-sensitive customers. |
| **2. Loss-Leader Governance** | Tables (-$17.7K) and Bookcases (-$3.5K) have over 31% loss rates. | Transition Tables and Bookcases to bundle-only pricing (e.g. Tables must be purchased with full-margin Chairs), and renegotiate supplier freight costs. | Improves Furniture margin from 2.5% towards the company benchmark of 12.5%. |
| **3. Account-Level Re-Pricing** | Customer Decile 10 generates $278K sales at -$65.8K net loss (average discount ~37%). | Enforce account-level margin floors and restrict recurring deep-discount concessions for chronic loss accounts. | Recovers bottom-line profitability from high-volume corporate accounts without cutting sales force headcount. |
| **4. Regional Policy Realignment** | Texas (-$25.7K) and Pennsylvania (-$15.6K) suffer under average promotional discounts >32%. | Review local pricing autonomy in Central and East territories to prevent regional price wars. | Elevates Central region margin from 7.9% closer to West region's 14.9%. |
